In [1]:
from pathlib import Path
import json

try:
    import chardet
except ImportError:
    print("Установите:")
    print("pip install chardet")
    raise


RAW_DIR = Path("./laws_raw")
OUT_DIR = Path("./mipt_tz_dataset/laws_processed")

OUT_DIR.mkdir(parents=True, exist_ok=True)


def read_any_encoding(path: Path) -> str:
    data = path.read_bytes()

    enc = chardet.detect(data)["encoding"]

    if enc is None:
        enc = "utf-8"

    try:
        return data.decode(enc)
    except Exception:
        for e in [
            "utf-8",
            "cp1251",
            "windows-1251",
            "utf-8-sig",
            "koi8-r",
            "cp866",
        ]:
            try:
                return data.decode(e)
            except Exception:
                pass

    return data.decode("utf-8", errors="ignore")


processed = []

for file in RAW_DIR.glob("*.txt"):

    text = read_any_encoding(file)

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    out_txt = OUT_DIR / file.name

    out_txt.write_text(
        text,
        encoding="utf-8"
    )

    meta = {
        "type": "law",
        "name": file.stem,
        "source_file": str(file),
        "processed_file": str(out_txt),
        "text_length": len(text),
    }

    (OUT_DIR / f"{file.stem}.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    processed.append(meta)

with open(
    OUT_DIR / "processed_index.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for item in processed:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


print(f"Обработано {len(processed)} документов.")

Обработано 7 документов.
